In [2]:
# Import relevant libraries 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from pathlib import Path
import zipfile
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from datetime import datetime

In [3]:
# Current working directory
current_dir = os.getcwd()
print(current_dir)

/workspaces/DSE3101-Project/backend


In [4]:
# Go up to repo root, then into data folder to retrive raw data
zip_path = os.path.join(current_dir, "..", "data", "raw", "ResaleFlatPrices.zip")
zip_path

'/workspaces/DSE3101-Project/backend/../data/raw/ResaleFlatPrices.zip'

In [5]:
all_dfs = []

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    csv_files = [f for f in zip_ref.namelist() if f.endswith('.csv')]
    
    for csv_file in csv_files:
        with zip_ref.open(csv_file) as f:
            df = pd.read_csv(f)
            all_dfs.append(df)

# Combine into one DataFrame
combined_df = pd.concat(all_dfs, ignore_index=True)

In [6]:
print("=" * 50)
print("STEP 1: INITIAL DATA INSPECTION")
print("=" * 50)

print(f"\nDataset shape: {combined_df.shape}")
print(f"\nColumn names:\n{combined_df.columns.tolist()}")
print(f"\nData types:\n{combined_df.dtypes}")
print(f"\nFirst few rows:\n{combined_df.head()}")
print(f"\nBasic statistics:\n{combined_df.describe()}")
print(f"\nUnique values per column:")
for col in combined_df.columns:
    print(f"  {col}: {combined_df[col].nunique()}")

STEP 1: INITIAL DATA INSPECTION

Dataset shape: (970969, 11)

Column names:
['month', 'town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'resale_price', 'remaining_lease']

Data types:
month                      str
town                       str
flat_type                  str
block                      str
street_name                str
storey_range               str
floor_area_sqm         float64
flat_model                 str
lease_commence_date      int64
resale_price           float64
remaining_lease         object
dtype: object

First few rows:
     month        town flat_type block       street_name storey_range  \
0  1990-01  ANG MO KIO    1 ROOM   309  ANG MO KIO AVE 1     10 TO 12   
1  1990-01  ANG MO KIO    1 ROOM   309  ANG MO KIO AVE 1     04 TO 06   
2  1990-01  ANG MO KIO    1 ROOM   309  ANG MO KIO AVE 1     10 TO 12   
3  1990-01  ANG MO KIO    1 ROOM   309  ANG MO KIO AVE 1     07 TO 09   
4  1990-01  A

In [7]:
# get current year 
current_year = datetime.now().year  

# Extract year from lease_commence_date and calculate
combined_df['lease_year'] = pd.to_datetime(combined_df['lease_commence_date']).dt.year
combined_df['remaining_lease'] = 99 - (current_year - combined_df['lease_year'])

# Clean up
combined_df = combined_df.drop('lease_year', axis=1)

print("✓ Added remaining_lease column")
print(f"Sample: {combined_df[['lease_commence_date', 'remaining_lease']].head()}")


✓ Added remaining_lease column
Sample:    lease_commence_date  remaining_lease
0                 1977               43
1                 1977               43
2                 1977               43
3                 1977               43
4                 1976               43


In [8]:
print("\n" + "=" * 80)
print("STEP 2: MISSING VALUE ANALYSIS")
print("=" * 80)

missing_summary = pd.DataFrame({
    'Missing_Count': combined_df.isnull().sum(),
    'Percentage': (combined_df.isnull().sum() / len(combined_df)) * 100,
    'Data_Type': combined_df.dtypes
})
print(missing_summary)

# Handle missing values
if combined_df.isnull().sum().sum() > 0:
    # For critical columns, drop rows
    critical_cols = ['resale_price', 'floor_area_sqm', 'lease_commence_date', 'town', 'flat_type']
    combined_df = combined_df.dropna(subset=critical_cols)
    print(f"\n✓ Dropped rows with missing critical values")
    print(f"New shape: {combined_df.shape}")


STEP 2: MISSING VALUE ANALYSIS
                     Missing_Count  Percentage Data_Type
month                            0         0.0       str
town                             0         0.0       str
flat_type                        0         0.0       str
block                            0         0.0       str
street_name                      0         0.0       str
storey_range                     0         0.0       str
floor_area_sqm                   0         0.0   float64
flat_model                       0         0.0       str
lease_commence_date              0         0.0     int64
resale_price                     0         0.0   float64
remaining_lease                  0         0.0     int32


In [9]:
print("\n" + "=" * 80)
print("STEP 3: DUPLICATE REMOVAL")
print("=" * 80)

initial_rows = len(combined_df)
duplicates = combined_df.duplicated().sum()
print(f"Duplicate rows found: {duplicates}")

print(combined_df[combined_df.duplicated(keep=False)].head())

if duplicates > 0:
    combined_df = combined_df.drop_duplicates()
    print(f"✓ Removed {initial_rows - len(combined_df)} duplicate rows")
    print(f"New shape: {combined_df.shape}")


STEP 3: DUPLICATE REMOVAL
Duplicate rows found: 2006
       month         town flat_type block    street_name storey_range  \
672  1990-01      GEYLANG    3 ROOM    47     CIRCUIT RD     01 TO 03   
673  1990-01      GEYLANG    3 ROOM    47     CIRCUIT RD     01 TO 03   
725  1990-01      HOUGANG    3 ROOM   308  HOUGANG AVE 5     10 TO 12   
726  1990-01      HOUGANG    3 ROOM   308  HOUGANG AVE 5     10 TO 12   
842  1990-01  JURONG WEST    3 ROOM   145    HU CHING RD     04 TO 06   

     floor_area_sqm      flat_model  lease_commence_date  resale_price  \
672            56.0        STANDARD                 1969       18000.0   
673            56.0        STANDARD                 1969       18000.0   
725            67.0  NEW GENERATION                 1983       47000.0   
726            67.0  NEW GENERATION                 1983       47000.0   
842            64.0        IMPROVED                 1976       23400.0   

     remaining_lease  
672               43  
673             

In [10]:
# ============================================
# STEP 4: DATA TYPE CONVERSION & PARSING
# ============================================
print("\n" + "=" * 80)
print("STEP 4: DATA TYPE CONVERSION & PARSING")
print("=" * 80)

# Convert month to datetime
combined_df['month'] = pd.to_datetime(combined_df['month'], format='%Y-%m')
print("✓ Converted 'month' to datetime")

# Find middle of storey_range (e.g., Split "01 TO 05" → get both numbers → average → single storey_mid column)
split_storey = combined_df['storey_range'].str.split(' TO ', expand=True).astype(int)
combined_df['storey_mid'] = split_storey[[0,1]].mean(axis=1)

# Ensure numeric columns are numeric
numeric_cols = ['block', 'floor_area', 'lease_commence', 'resale_price']
for col in numeric_cols:
    if col in combined_df.columns:
        combined_df[col] = pd.to_numeric(combined_df[col], errors='coerce')

print("✓ Ensured numeric columns have correct data types")


STEP 4: DATA TYPE CONVERSION & PARSING
✓ Converted 'month' to datetime
✓ Ensured numeric columns have correct data types


In [11]:
# ============================================
# STEP 5: OUTLIER DETECTION & HANDLING
# ============================================
print("\n" + "=" * 80)
print("STEP 5: OUTLIER DETECTION & HANDLING")
print("=" * 80)

def detect_outliers_iqr(df, column, multiplier=1.5):
    """Detect outliers using IQR method"""
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - multiplier * IQR
    upper_bound = Q3 + multiplier * IQR
    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    return outliers, lower_bound, upper_bound

# Check key columns for outliers
outlier_cols = ['resale_price', 'floor_area_sqm']
outlier_stats = {}

for col in outlier_cols:
    if col in combined_df.columns:
        outliers, lower, upper = detect_outliers_iqr(combined_df, col)
        outlier_stats[col] = {
            'count': len(outliers),
            'percentage': len(outliers) / len(combined_df) * 100,
            'lower_bound': lower,
            'upper_bound': upper
        }
        print(f"\n{col}:")
        print(f"  Outliers: {len(outliers)} ({len(outliers)/len(combined_df)*100:.2f}%)")
        print(f"  Valid range: {lower:.2f} to {upper:.2f}")
        print(f"  Min: {combined_df[col].min()}, Max: {combined_df[col].max()}")

# For this project, keep outliers as they may represent genuine luxury/large flats
# But flag them for analysis
combined_df['is_outlier_price'] = 0
outliers, lower, upper = detect_outliers_iqr(combined_df, 'resale_price')
combined_df.loc[outliers.index, 'is_outlier_price'] = 1
print(f"\n✓ Flagged {outliers.shape[0]} price outliers for analysis")


STEP 5: OUTLIER DETECTION & HANDLING

resale_price:
  Outliers: 24117 (2.49%)
  Valid range: -152500.00 to 787500.00
  Min: 5000.0, Max: 1658888.0

floor_area_sqm:
  Outliers: 2483 (0.26%)
  Valid range: 13.00 to 173.00
  Min: 28.0, Max: 366.7

✓ Flagged 24117 price outliers for analysis


In [15]:
print("\n" + "=" * 80)
print("STEP 6: FEATURE ENGINEERING")
print("=" * 80)

# Storey categories
combined_df['storey_category'] = pd.cut(combined_df['storey_mid'],
                                        bins=[0, 5, 10, 15, 50],
                                        labels=['Low', 'Mid-Low', 'Mid-High', 'High'])

print("✓ Created floor_area_category")

# Categorize towns into regions (based on Singapore planning areas)
region_mapping = {
    'ANG MO KIO': 'North-East',
    'BEDOK': 'East',
    'BISHAN': 'Central',
    'BUKIT BATOK': 'West',
    'BUKIT MERAH': 'Central',
    'BUKIT PANJANG': 'West',
    'BUKIT TIMAH': 'Central',
    'CENTRAL AREA': 'Central',
    'CHOA CHU KANG': 'West',
    'CLEMENTI': 'West',
    'GEYLANG': 'Central',
    'HOUGANG': 'North-East',
    'JURONG EAST': 'West',
    'JURONG WEST': 'West',
    'KALLANG/WHAMPOA': 'Central',
    'MARINE PARADE': 'Central',
    'PASIR RIS': 'East',
    'PUNGGOL': 'North-East',
    'QUEENSTOWN': 'Central',
    'SEMBAWANG': 'North',
    'SENGKANG': 'North-East',
    'SERANGOON': 'North-East',
    'TAMPINES': 'East',
    'TOA PAYOH': 'Central',
    'WOODLANDS': 'North',
    'YISHUN': 'North'
}

combined_df['region'] = combined_df['town'].map(region_mapping)
combined_df['region'] = combined_df['region'].fillna('Other')
print("✓ Created region feature")

# Flag mature vs non-mature estates
mature_estates = ['ANG MO KIO', 'BEDOK', 'BISHAN', 'BUKIT MERAH', 'BUKIT TIMAH',
                  'CENTRAL AREA', 'CLEMENTI', 'GEYLANG', 'KALLANG/WHAMPOA',
                  'MARINE PARADE', 'QUEENSTOWN', 'SERANGOON', 'TOA PAYOH']

combined_df['is_mature_estate'] = combined_df['town'].isin(mature_estates).astype(int)
print("✓ Created is_mature_estate flag")


STEP 6: FEATURE ENGINEERING
✓ Created floor_area_category
✓ Created region feature
✓ Created is_mature_estate flag


In [17]:
# ============================================
# STEP 7: AGGREGATED FEATURES (TOWN-LEVEL STATISTICS)
# ============================================
print("\n" + "=" * 80)
print("STEP 11: AGGREGATED FEATURES (TOWN-LEVEL STATISTICS)")
print("=" * 80)

# Town-level median price (useful for understanding relative value)
town_median_price = combined_df.groupby('town')['resale_price'].median()
combined_df['town_median_price'] = combined_df['town'].map(town_median_price)
combined_df['price_vs_town_median'] = combined_df['resale_price'] / combined_df['town_median_price']
print("✓ Created town_median_price")


STEP 11: AGGREGATED FEATURES (TOWN-LEVEL STATISTICS)
✓ Created town_median_price


In [23]:
# Remove negative or zero prices
print("\n1. Validating resale_price...")
print(f"   Min price: ${combined_df['resale_price'].min():,.0f}")
print(f"   Max price: ${combined_df['resale_price'].max():,.0f}")

invalid_prices = combined_df[combined_df['resale_price'] <= 0]
print(f"   Found {len(invalid_prices)} records with price <= 0")
combined_df = combined_df[combined_df['resale_price'] > 0]

# Rule 2: Floor area must be positive and realistic
print("\n2. Validating floor_area...")
print(f"   Min area: {combined_df['floor_area_sqm'].min()} sqm")
print(f"   Max area: {combined_df['floor_area_sqm'].max()} sqm")

invalid_area = combined_df[combined_df['floor_area_sqm'] <= 0]
print(f"   Found {len(invalid_area)} records with area <= 0")
combined_df = combined_df[combined_df['floor_area_sqm'] > 0]


1. Validating resale_price...
   Min price: $5,000
   Max price: $1,658,888
   Found 0 records with price <= 0

2. Validating floor_area...
   Min area: 28.0 sqm
   Max area: 366.7 sqm
   Found 0 records with area <= 0


In [26]:
combined_df['year'] = pd.to_datetime(combined_df['month']).dt.year

In [28]:
print("\n" + "=" * 80)
print("INFLATION ADJUSTMENT TO 2008 BASE YEAR")
print("=" * 80)

# Singapore Consumer Price Index (CPI) data
# Source: Department of Statistics Singapore (SingStat)
# Base year: 2008 = 100
singapore_cpi = {
    1990: 65.8,
    1991: 68.5,
    1992: 70.9,
    1993: 73.1,
    1994: 75.9,
    1995: 77.6,
    1996: 79.1,
    1997: 81.0,
    1998: 80.5,
    1999: 80.7,
    2000: 82.1,
    2001: 83.1,
    2002: 82.6,
    2003: 83.1,
    2004: 84.8,
    2005: 85.3,
    2006: 86.3,
    2007: 88.5,
    2008: 100.0,  # Base year
    2009: 100.6,
    2010: 103.4,
    2011: 108.6,
    2012: 113.2,
    2013: 115.9,
    2014: 117.0,
    2015: 116.6,
    2016: 116.5,
    2017: 117.2,
    2018: 117.9,
    2019: 118.5,
    2020: 118.2,
    2021: 120.8,
    2022: 127.0,
    2023: 132.8,
    2024: 136.5,  # Estimated
    2025: 140.0,  # Projected
    2026: 143.0   # Projected
}

# Map CPI to each transaction year
print("\n1. Mapping CPI values to transaction years...")
combined_df['cpi'] = combined_df['year'].map(singapore_cpi)

# Check for any missing CPI values
missing_cpi = combined_df[combined_df['cpi'].isna()]
if len(missing_cpi) > 0:
    print(f"   ⚠️  Warning: {len(missing_cpi)} records have missing CPI values")
    print(f"   Years with missing CPI: {missing_cpi['year'].unique()}")
    
    # Fill missing CPI with nearest available year
    for year in missing_cpi['year'].unique():
        if year < min(singapore_cpi.keys()):
            combined_df.loc[combined_df['year'] == year, 'cpi'] = singapore_cpi[min(singapore_cpi.keys())]
            print(f"   Used {min(singapore_cpi.keys())} CPI for year {year}")
        elif year > max(singapore_cpi.keys()):
            combined_df.loc[combined_df['year'] == year, 'cpi'] = singapore_cpi[max(singapore_cpi.keys())]
            print(f"   Used {max(singapore_cpi.keys())} CPI for year {year}")
else:
    print("   ✓ All years have CPI values")

# Calculate inflation-adjusted (real) price
# Formula: Real Price = Nominal Price × (Base Year CPI / Current Year CPI)
base_year_cpi = 100.0  # 2008 CPI

print("\n2. Calculating inflation-adjusted prices...")
combined_df['resale_price_real_2008'] = (
    combined_df['resale_price'] * (base_year_cpi / combined_df['cpi'])
)

# Round to nearest dollar
combined_df['resale_price_real_2008'] = combined_df['resale_price_real_2008'].round(2)

print("✓ Created resale_price_real_2008 (inflation-adjusted to 2008 dollars)")

# Calculate the adjustment factor for reference
combined_df['inflation_adjustment_factor'] = base_year_cpi / combined_df['cpi']

# Show examples
print("\n3. Sample of inflation adjustment:")
print("="*100)
sample_years = combined_df.groupby('year').first()[
    ['resale_price', 'cpi', 'inflation_adjustment_factor', 'resale_price_real_2008']
].head(10)
print(sample_years)

# Calculate statistics
print("\n4. Price comparison statistics:")
print("="*100)
print(f"Nominal (Original) Prices:")
print(f"  Mean:   ${combined_df['resale_price'].mean():,.2f}")
print(f"  Median: ${combined_df['resale_price'].median():,.2f}")
print(f"  Min:    ${combined_df['resale_price'].min():,.2f}")
print(f"  Max:    ${combined_df['resale_price'].max():,.2f}")

print(f"\nReal (2008-adjusted) Prices:")
print(f"  Mean:   ${combined_df['resale_price_real_2008'].mean():,.2f}")
print(f"  Median: ${combined_df['resale_price_real_2008'].median():,.2f}")
print(f"  Min:    ${combined_df['resale_price_real_2008'].min():,.2f}")
print(f"  Max:    ${combined_df['resale_price_real_2008'].max():,.2f}")

# Calculate real price change over time
print("\n5. Real vs Nominal price trends by year:")
print("="*100)
yearly_comparison = combined_df.groupby('year').agg({
    'resale_price': 'median',
    'resale_price_real_2008': 'median',
    'cpi': 'first'
}).round(2)

yearly_comparison['real_vs_nominal_diff'] = (
    yearly_comparison['resale_price_real_2008'] - yearly_comparison['resale_price']
)
yearly_comparison['real_vs_nominal_pct'] = (
    (yearly_comparison['resale_price_real_2008'] / yearly_comparison['resale_price'] - 1) * 100
).round(2)

print(yearly_comparison.head(30))


INFLATION ADJUSTMENT TO 2008 BASE YEAR

1. Mapping CPI values to transaction years...
   ✓ All years have CPI values

2. Calculating inflation-adjusted prices...
✓ Created resale_price_real_2008 (inflation-adjusted to 2008 dollars)

3. Sample of inflation adjustment:
      resale_price   cpi  inflation_adjustment_factor  resale_price_real_2008
year                                                                         
1990        9000.0  65.8                     1.519757                13677.81
1991        6000.0  68.5                     1.459854                 8759.12
1992        7800.0  70.9                     1.410437                11001.41
1993       57000.0  73.1                     1.367989                77975.38
1994       24000.0  75.9                     1.317523                31620.55
1995       90000.0  77.6                     1.288660               115979.38
1996       48000.0  79.1                     1.264223                60682.68
1997      118000.0  81.0     